# LeetCode 481 — Magical String bằng Python

Link đề: https://leetcode.com/problems/magical-string/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Mô phỏng chuỗi tự mô tả (dãy Kolakoski)

Skeleton LeetCode:

```python
class Solution:
    def magicalString(self, n: int) -> int:
        ...
```

Ghi chú: bài này là bài thường (không Premium). Đề được viết lại bằng tiếng Việt theo đề gốc; các ví dụ phụ do notebook tự dựng sẽ được ghi rõ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu thế nào là **chuỗi "tự mô tả"** (magical string): các độ dài nhóm liên tiếp của chính chuỗi lại sinh ra đúng chuỗi đó.
2. Dùng được các công cụ Python: nhân danh sách `[2] * 3`, cộng danh sách bằng `+=`, cắt lát `s[:n]`, `list.count`, `itertools.groupby`, generator (`yield`) và `itertools.islice`.
3. Nắm kỹ thuật **hai con trỏ trên cùng một mảng**: con trỏ **ghi** (phần đang mở rộng) chạy nhanh hơn con trỏ **đọc** (chỉ vào độ dài của nhóm kế tiếp).
4. Cài đặt 4 cách khác nhau về ý tưởng: **vét cạn mọi chuỗi 1/2 rồi lọc theo tính chất tự mô tả → mô phỏng hai con trỏ → tiền xử lý một lần + mảng đếm tiền tố → bộ sinh lồng nhau dùng bộ nhớ O(log n)**.
5. Tránh các lỗi kinh điển: quên cắt `s[:n]` khi nhóm cuối vượt quá `n`, lấy ký tự xen kẽ sai, đặt sai con trỏ đọc ban đầu, và biết cách kiểm thử bài toán bằng một **hàm kiểm tra tính chất** thay vì chỉ so đáp án.

## 2. Đề bài & ví dụ

Một chuỗi **kỳ diệu** `s` chỉ gồm ký tự `'1'` và `'2'` và thỏa quy tắc:

- Nếu ta **gom các ký tự giống nhau liên tiếp** của `s` thành từng nhóm, rồi ghi lại **độ dài của từng nhóm**, thì dãy các độ dài đó nối lại **chính là `s`**.

Các ký tự đầu của `s` là `"1221121221221121122..."`. Gom nhóm:

```
1 | 22 | 11 | 2 | 1 | 22 | 1 | 22 | 11 | 2 | 11 | 22 ...
```

Độ dài các nhóm: `1 2 2 1 1 2 1 2 2 1 2 2 ...` — nối lại đúng bằng `1221121221221...` chính là `s`.

Cho số nguyên `n`, hãy trả về **số lượng ký tự `1`** trong `n` ký tự đầu tiên của `s`.

**Ví dụ 1:** `n = 6` → `3`. Sáu ký tự đầu là `"122112"`, chứa ba ký tự `1`.

**Ví dụ 2:** `n = 1` → `1`. Ký tự đầu là `"1"`.

**Ràng buộc:** `1 <= n <= 10^5`.

In [1]:
mau = "1221121221221121122"          # 19 ký tự đầu của chuỗi kỳ diệu (theo đề)
for n, kq in [(6, 3), (1, 1)]:
    print(f"n = {n}: n ký tự đầu = {mau[:n]!r}, số '1' = {mau[:n].count('1')}, kỳ vọng = {kq}")
    assert mau[:n].count("1") == kq

n = 6: n ký tự đầu = '122112', số '1' = 3, kỳ vọng = 3
n = 1: n ký tự đầu = '1', số '1' = 1, kỳ vọng = 1


## 3. Kiến thức nền

### 3.1. Nhân danh sách và cộng danh sách

`[2] * 3` ra `[2, 2, 2]`: lặp danh sách 3 lần. Toán tử `+=` trên danh sách **nối thêm** các phần tử vào cuối danh sách hiện tại (nó sửa tại chỗ). Hai thứ này kết hợp thành mẹo "ghi `k` ký tự giống nhau": `s += [ky_tu] * k`.

In [2]:
print([2] * 3)                 # [2, 2, 2]
s = [1, 2, 2]
s += [1] * 2                   # nối thêm hai số 1
print(s)                       # [1, 2, 2, 1, 1]
print(len(s))                  # 5

[2, 2, 2]
[1, 2, 2, 1, 1]
5


### 3.2. Chỉ số âm, cắt lát `s[:n]` và `count`

`s[-1]` là phần tử **cuối cùng** của danh sách. `s[:n]` lấy `n` phần tử đầu (nếu danh sách dài hơn `n` thì phần dư bị bỏ). `s.count(1)` đếm số lần xuất hiện của `1`. Ta sẽ cần `s[:n]` vì khi ghi cả nhóm, danh sách có thể **dài quá `n`**.

In [3]:
s = [1, 2, 2, 1, 1]
print(s[-1])                   # 1  (phần tử cuối)
print(s[:4])                   # [1, 2, 2, 1]
print(s.count(1), s[:4].count(1))   # 3 2 : đếm cả danh sách khác đếm n phần tử đầu

1
[1, 2, 2, 1]
3 2


### 3.3. Đảo ký tự xen kẽ bằng `3 - x`

Các nhóm của chuỗi **xen kẽ** `1`, `2`, `1`, `2`, ... Nếu ký tự của nhóm trước là `x` (chỉ `1` hoặc `2`), thì nhóm sau là `3 - x`: `3 - 1 = 2`, `3 - 2 = 1`. Đây là mẹo nhỏ thay cho `if x == 1: ... else: ...`.

In [4]:
x = 1
for _ in range(5):
    print(x, end=" ")
    x = 3 - x                  # 1 -> 2 -> 1 -> 2 ...
print()

1 2 1 2 1 


### 3.4. Gom nhóm bằng `itertools.groupby` (để kiểm chứng đề bài)

`groupby(iterable)` gom các phần tử **giống nhau liền kề**. Mỗi nhóm cho ta `(khóa, các phần tử)`; `len(list(nhom))` là độ dài nhóm. Ta dùng nó để **kiểm tra lại định nghĩa** trên mẫu của đề.

In [5]:
from itertools import groupby

def do_dai_cac_nhom(s):
    """Độ dài các nhóm ký tự giống nhau liên tiếp của chuỗi s."""
    return [len(list(g)) for _, g in groupby(s)]

print(do_dai_cac_nhom("1221121221221121122"))
# [1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2]  -> trùng với đầu chuỗi!
print("".join(map(str, do_dai_cac_nhom(mau))) , "vs", mau)

[1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2]
122112122122 vs 1221121221221121122


### 3.5. Generator (`yield`) và `islice`

Hàm có `yield` không trả hết một lần mà **phát từng giá trị khi được hỏi** (`next(...)` hoặc vòng `for`). `itertools.islice(gen, n)` lấy `n` giá trị đầu của generator rồi dừng. Vì generator sinh theo nhu cầu nên có thể "vô hạn" mà vẫn không tốn bộ nhớ. Ta cần ý này cho Cách 4.

In [6]:
from itertools import islice

def dem_so():
    k = 1
    while True:                # vô hạn nhưng chỉ chạy khi được hỏi
        yield k
        k += 1

print(list(islice(dem_so(), 5)))     # [1, 2, 3, 4, 5]
g = dem_so()
print(next(g), next(g), next(g))     # 1 2 3

[1, 2, 3, 4, 5]
1 2 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Chuỗi bắt đầu bằng ký tự `1` (đề cho). Ta tự dựng từng bước:

- **Nhóm 0** gồm ký tự `1` và có độ dài bằng `s[0] = 1` (vì `s[0]` là `1`). Vậy ta có `s = "1"`.
- **Nhóm 1** gồm ký tự `2` (xen kẽ với nhóm 0), độ dài bằng `s[1]`. Nhưng `s[1]` chính là **ký tự đầu của nhóm 1**, mà nhóm 1 toàn ký tự `2` nên `s[1] = 2`. Vậy nhóm 1 là `"22"` và `s = "122"`.
- **Nhóm 2** gồm ký tự `1`, độ dài bằng `s[2] = 2` ⇒ `"11"`. `s = "12211"`.
- **Nhóm 3** gồm ký tự `2`, độ dài `s[3] = 1` ⇒ `"2"`. `s = "122112"`.
- **Nhóm 4** gồm ký tự `1`, độ dài `s[4] = 1` ⇒ `"1"`. `s = "1221121"`.
- **Nhóm 5** gồm ký tự `2`, độ dài `s[5] = 2` ⇒ `"22"`. `s = "122112122"`.

Quan sát then chốt: để ghi nhóm `k`, ta cần `s[k]`, mà `s[k]` luôn đã được ghi **từ trước** vì chuỗi mọc nhanh hơn con trỏ `k` (mỗi nhóm có độ dài ít nhất 1 và mỗi bước đọc chỉ tiêu thụ một ký tự). Chuỗi vừa là **dữ liệu**, vừa là **chương trình** điều khiển chính nó.

Ví dụ `n = 6`: sáu ký tự đầu `"122112"` ⇒ đếm được ba chữ `1` (vị trí 0, 3, 4).

In [7]:
s = [1]
print("sau nhóm 0:", s)
s += [2] * 2              # nhóm 1: ký tự 2, độ dài s[1] sẽ là 2
print("sau nhóm 1:", s)
s += [1] * s[2]           # nhóm 2: ký tự 1, độ dài s[2]
print("sau nhóm 2:", s)
s += [2] * s[3]           # nhóm 3: ký tự 2, độ dài s[3]
print("sau nhóm 3:", s)
s += [1] * s[4]           # nhóm 4
print("sau nhóm 4:", s)
s += [2] * s[5]           # nhóm 5
print("sau nhóm 5:", s, "->", "".join(map(str, s)))

sau nhóm 0: [1]
sau nhóm 1: [1, 2, 2]
sau nhóm 2: [1, 2, 2, 1, 1]
sau nhóm 3: [1, 2, 2, 1, 1, 2]
sau nhóm 4: [1, 2, 2, 1, 1, 2, 1]
sau nhóm 5: [1, 2, 2, 1, 1, 2, 1, 2, 2] -> 122112122


## 5. Cách 1 — Brute Force / baseline: thử mọi chuỗi 1/2 và lọc theo định nghĩa

**Ý tưởng:** dựa **thẳng vào định nghĩa**. Chuỗi cần tìm là chuỗi `n` ký tự `1`/`2` mà "tự mô tả". Có `2^n` chuỗi như vậy, ta thử hết và giữ lại chuỗi thỏa tính chất.

Vì ta chỉ có `n` ký tự đầu của chuỗi vô hạn, tính chất phải xét theo **tiền tố**: gom `c` thành các nhóm; với mọi nhóm **không phải nhóm cuối**, độ dài nhóm thứ `t` phải đúng bằng `c[t]`; nhóm cuối có thể bị **cắt ngắn** bởi `n` nên chỉ cần độ dài của nó **không vượt quá** `c[t]`. Ngoài ra `c[0] = 1` (đề cho).

Cách này **hoàn toàn độc lập** với phép mô phỏng nên rất hợp làm **oracle** (đáp án đối chứng) cho các cách sau, dù chỉ chạy được với `n` nhỏ.

In [8]:
from itertools import product

def la_tien_to_hop_le(c):
    """c là tuple gồm 1/2. Kiểm tra c có thể là tiền tố của chuỗi kỳ diệu hay không."""
    if c[0] != 1:
        return False
    nhom = [len(list(g)) for _, g in groupby(c)]
    m = len(nhom)
    for t in range(m):
        if t < m - 1:
            if nhom[t] != c[t]:        # nhóm đầy đủ phải đúng bằng c[t]
                return False
        else:
            if nhom[t] > c[t]:         # nhóm cuối có thể bị cắt ngắn nhưng không được dài hơn c[t]
                return False
    return True

def dem_vet_can(n):
    hop_le = [c for c in product((1, 2), repeat=n) if la_tien_to_hop_le(c)]
    assert len(hop_le) == 1, hop_le       # chuỗi n ký tự đầu là DUY NHẤT
    return hop_le[0].count(1), "".join(map(str, hop_le[0]))

for n in (1, 2, 3, 6, 10, 14):
    print(n, dem_vet_can(n))

1 (1, '1')
2 (1, '12')
3 (1, '122')
6 (3, '122112')
10 (5, '1221121221')
14 (7, '12211212212211')


**Chạy các ví dụ của đề:**

In [9]:
assert dem_vet_can(6)[0] == 3
assert dem_vet_can(1)[0] == 1
print("Ví dụ 1 và 2 đều đúng với vét cạn.")

Ví dụ 1 và 2 đều đúng với vét cạn.


**Độ phức tạp:** `2^n` chuỗi, mỗi chuỗi kiểm tra `O(n)` ⇒ `O(n · 2^n)` thời gian, `O(n)` bộ nhớ cho một chuỗi. Với `n = 10^5` là hoàn toàn bất khả thi.

**Nút thắt:** ta đã *thấy* ở mục 4 rằng chuỗi được **xác định duy nhất** bởi chính nó — đã có tiền tố thì ký tự kế tiếp bị ép. Vậy đáng lẽ không cần thử `2^n` khả năng; chỉ cần **dựng** chuỗi một mạch.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Chuỗi tự xác định.** Biết tiền tố `s[0..L-1]` và con trỏ đọc `i`, nhóm kế tiếp có **ký tự** `3 - s[-1]` (xen kẽ) và **độ dài** `s[i]`. Không có lựa chọn nào khác ⇒ không cần thử `2^n`.
2. **Hai con trỏ trên cùng một mảng:** con trỏ ghi = `len(s)` (cuối mảng), con trỏ đọc `i`. Luôn có `i < len(s)` vì mỗi nhóm ghi **ít nhất 1** ký tự mà đọc chỉ tiến **1** ô; khởi đầu `s = [1, 2, 2]` đã có `i = 2 < 3`.
3. **Khởi tạo `s = [1, 2, 2]` và `i = 2`:** hai nhóm đầu (`"1"` và `"22"`) đã biết nên ta bắt đầu từ nhóm thứ 3 (chỉ số 2), mà độ dài của nó là `s[2] = 2`.
4. **Dừng khi `len(s) >= n`**, rồi **cắt `s[:n]`** (nhóm cuối có thể ghi quá `n`) và đếm số `1`.
5. Mỗi vòng thêm ít nhất 1 phần tử ⇒ `O(n)` vòng, tổng việc ghi `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng hai con trỏ trên một danh sách (cách chuẩn)

**Giải thích từng đoạn:**

- `s = [1, 2, 2]`, `i = 2`: ba ký tự đầu đã biết; `i` chỉ vào phần tử cho biết độ dài nhóm tiếp theo.
- `while len(s) < n:` còn thiếu ký tự thì ghi tiếp.
- `cur = 3 - s[-1]`: ký tự nhóm mới **khác** ký tự cuối đã ghi (xen kẽ).
- `s += [cur] * s[i]`: ghi `s[i]` ký tự `cur`.
- `i += 1`: nhóm kế tiếp.
- `return s[:n].count(1)`: chỉ đếm `n` ký tự đầu.

In [10]:
def dem_hai_con_tro(n):
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        cur = 3 - s[-1]
        s += [cur] * s[i]
        i += 1
    return s[:n].count(1)

print(dem_hai_con_tro(6), dem_hai_con_tro(1))   # 3 1
for n in range(1, 15):
    assert dem_hai_con_tro(n) == dem_vet_can(n)[0]
print("Hai con trỏ khớp vét cạn với n = 1..14")

3 1
Hai con trỏ khớp vét cạn với n = 1..14


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def magicalString(self, n: int) -> int:
        s = [1, 2, 2]
        i = 2
        while len(s) < n:
            cur = 3 - s[-1]
            s += [cur] * s[i]
            i += 1
        return s[:n].count(1)
```

**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(n)`.

### Cách 3 — Tiền xử lý một lần + mảng đếm tiền tố (trả lời nhiều truy vấn)

**Ý tưởng khác:** trong LeetCode, hàm `magicalString` có thể bị gọi **nhiều lần** với nhiều `n` khác nhau (tối đa `10^5`). Thay vì dựng lại chuỗi mỗi lần, ta **dựng một lần** tới độ dài `10^5`, đồng thời lưu mảng `pref[k]` = số chữ `1` trong `k` ký tự đầu. Mỗi truy vấn sau đó là `O(1)`.

**Giải thích từng đoạn:**

- Dựng `s` đến ít nhất `LIMIT` ký tự (như Cách 2) rồi cắt `s[:LIMIT]`.
- `pref = [0]`; mỗi ký tự `x`: `pref.append(pref[-1] + (x == 1))`. (`True` cộng như `1`.)
- Truy vấn: `return pref[n]`.
- Cache ở **cấp lớp** để các lần gọi sau không dựng lại.

In [11]:
def xay_tien_to(limit):
    s = [1, 2, 2]
    i = 2
    while len(s) < limit:
        s += [3 - s[-1]] * s[i]
        i += 1
    s = s[:limit]
    pref = [0]
    for x in s:
        pref.append(pref[-1] + (x == 1))      # True được cộng như 1
    return pref

PREF = xay_tien_to(100000)
def dem_tien_to(n):
    return PREF[n]

print(dem_tien_to(6), dem_tien_to(1), dem_tien_to(100000))
for n in (1, 2, 3, 6, 19, 1000, 54321, 100000):
    assert dem_tien_to(n) == dem_hai_con_tro(n)
print("Khớp Cách 2 ở nhiều n")

3 1 49972
Khớp Cách 2 ở nhiều n


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    _pref = None

    def magicalString(self, n: int) -> int:
        if Solution._pref is None:
            limit = 100000
            s = [1, 2, 2]
            i = 2
            while len(s) < limit:
                s += [3 - s[-1]] * s[i]
                i += 1
            pref = [0]
            for x in s[:limit]:
                pref.append(pref[-1] + (x == 1))
            Solution._pref = pref
        return Solution._pref[n]
```

**Độ phức tạp:** dựng một lần `O(L)` với `L = 10^5`; mỗi truy vấn `O(1)`; bộ nhớ `O(L)`.

### Cách 4 — Bộ sinh lồng nhau, bộ nhớ `O(log n)`

**Ý tưởng khác:** để ghi tới vị trí `L` ta chỉ **đọc** tới khoảng `L / 1.5` (mỗi nhóm trung bình dài `1.5`). Mà đọc tới `L / 1.5` thì chính chuỗi đó cũng cần **một bản đọc** lùi sau ở `L / 1.5²`... Vậy ta không cần lưu cả chuỗi: dùng một generator, và con trỏ **đọc** của nó là **một generator khác cùng loại** (một "bản sao" của chính nó, được tạo khi cần). Chuỗi bản sao lồng nhau chỉ sâu `O(log n)`.

**Giải thích từng đoạn:**

- Ba ký tự đầu `1, 2, 2` được **phát cố định**.
- Sau đó tạo `doc = chuoi_ky_dieu()` (generator con) và **bỏ qua hai giá trị đầu** (`s[0]`, `s[1]` đã dùng để dựng nhóm 0 và nhóm 1).
- Với mỗi `x` do `doc` phát ra (độ dài nhóm), phát `x` lần ký tự `cur`, rồi đổi `cur = 3 - cur`.
- Generator con chỉ tạo bản sao con của nó khi đã phát 3 giá trị cố định, nên không bị đệ quy vô hạn.

In [12]:
def chuoi_ky_dieu():
    yield 1
    yield 2
    yield 2
    doc = chuoi_ky_dieu()          # "bản đọc" lùi sau: chính là chuỗi kỳ diệu
    next(doc)                      # bỏ s[0]
    next(doc)                      # bỏ s[1]
    cur = 1                        # nhóm kế tiếp (chỉ số 2) gồm ký tự 1
    for x in doc:                  # x = s[2], s[3], ... là độ dài các nhóm kế tiếp
        for _ in range(x):
            yield cur
        cur = 3 - cur

print(list(islice(chuoi_ky_dieu(), 19)))
print("".join(map(str, islice(chuoi_ky_dieu(), 19))) == mau)

def dem_bo_sinh(n):
    return sum(1 for x in islice(chuoi_ky_dieu(), n) if x == 1)

print(dem_bo_sinh(6), dem_bo_sinh(1))
for n in (1, 2, 3, 6, 19, 1000, 54321, 100000):
    assert dem_bo_sinh(n) == dem_hai_con_tro(n)
print("Khớp Cách 2 ở nhiều n")

[1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2, 1, 1, 2, 1, 1, 2, 2]
True
3 1


Khớp Cách 2 ở nhiều n


**Code nộp LeetCode (Cách 4):**

```python
from itertools import islice

class Solution:
    def magicalString(self, n: int) -> int:
        def gen():
            yield 1
            yield 2
            yield 2
            doc = gen()
            next(doc)
            next(doc)
            cur = 1
            for x in doc:
                for _ in range(x):
                    yield cur
                cur = 3 - cur
        return sum(1 for x in islice(gen(), n) if x == 1)
```

**Độ phức tạp:** thời gian `O(n)` (tổng số giá trị phát ở mọi tầng là cấp số nhân giảm công bội `~2/3`, tức `~3n`); bộ nhớ `O(log n)` vì chỉ có `~log_{1.5} n` generator lồng nhau (độ sâu `~26` với `n = 10^5`).

In [13]:
# Đếm độ sâu lồng nhau của các generator: mỗi tầng phải lùi sau ~1.5 lần
import math
for n in (10, 1000, 100000):
    print(f"n = {n:>6}: sâu khoảng {math.log(n / 3, 1.5):.1f} tầng")

n =     10: sâu khoảng 3.0 tầng
n =   1000: sâu khoảng 14.3 tầng
n = 100000: sâu khoảng 25.7 tầng


## 8. TRACE từng bước

Dùng Cách 2 với `n = 12`. Cột: `i` = con trỏ đọc, `s[i]` = độ dài nhóm cần ghi, `cur` = ký tự nhóm, `len(s)` sau khi ghi.

| bước | i | s[i] | cur | thêm vào | s sau bước | len |
|---|---|---|---|---|---|---|
| bắt đầu | 2 |  |  |  | 1 2 2 | 3 |
| 1 | 2 | 2 | 1 | 1 1 | 1 2 2 1 1 | 5 |
| 2 | 3 | 1 | 2 | 2 | 1 2 2 1 1 2 | 6 |
| 3 | 4 | 1 | 1 | 1 | 1 2 2 1 1 2 1 | 7 |
| 4 | 5 | 2 | 2 | 2 2 | 1 2 2 1 1 2 1 2 2 | 9 |
| 5 | 6 | 1 | 1 | 1 | 1 2 2 1 1 2 1 2 2 1 | 10 |
| 6 | 7 | 2 | 2 | 2 2 | 1 2 2 1 1 2 1 2 2 1 2 2 | 12 |

`len(s) = 12 >= n` ⇒ dừng. Đếm `s[:12]` được **5** chữ `1` (vị trí 0, 3, 4, 6, 9).

In [14]:
def trace_hai_con_tro(n):
    s = [1, 2, 2]
    i = 2
    print(f"bắt đầu: s = {s}, i = {i}")
    buoc = 0
    while len(s) < n:
        buoc += 1
        cur = 3 - s[-1]
        them = [cur] * s[i]
        print(f"bước {buoc}: i={i}, s[i]={s[i]}, cur={cur}, thêm {them} -> ", end="")
        s += them
        i += 1
        print(f"s = {s} (len={len(s)})")
    print(f"dừng vì len(s)={len(s)} >= n={n}")
    print(f"s[:n] = {s[:n]} -> số 1 = {s[:n].count(1)}")
    return s[:n].count(1)

assert trace_hai_con_tro(12) == 5

bắt đầu: s = [1, 2, 2], i = 2
bước 1: i=2, s[i]=2, cur=1, thêm [1, 1] -> s = [1, 2, 2, 1, 1] (len=5)
bước 2: i=3, s[i]=1, cur=2, thêm [2] -> s = [1, 2, 2, 1, 1, 2] (len=6)
bước 3: i=4, s[i]=1, cur=1, thêm [1] -> s = [1, 2, 2, 1, 1, 2, 1] (len=7)
bước 4: i=5, s[i]=2, cur=2, thêm [2, 2] -> s = [1, 2, 2, 1, 1, 2, 1, 2, 2] (len=9)
bước 5: i=6, s[i]=1, cur=1, thêm [1] -> s = [1, 2, 2, 1, 1, 2, 1, 2, 2, 1] (len=10)
bước 6: i=7, s[i]=2, cur=2, thêm [2, 2] -> s = [1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2] (len=12)
dừng vì len(s)=12 >= n=12
s[:n] = [1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2] -> số 1 = 5


**Trace trường hợp ghi quá `n`:** với `n = 4`, sau bước 1 danh sách dài 5 > 4 nên phải cắt.

In [15]:
assert trace_hai_con_tro(4) == 2      # "1221" có hai chữ 1; nếu đếm cả danh sách dài 5 sẽ ra 3 (sai)

bắt đầu: s = [1, 2, 2], i = 2
bước 1: i=2, s[i]=2, cur=1, thêm [1, 1] -> s = [1, 2, 2, 1, 1] (len=5)
dừng vì len(s)=5 >= n=4
s[:n] = [1, 2, 2, 1] -> số 1 = 2


## 9. Code nộp LeetCode

Đây là bản gọn nhất (Cách 2), đúng `class Solution`, đúng tên phương thức và chữ ký:

In [16]:
class Solution:
    def magicalString(self, n: int) -> int:
        s = [1, 2, 2]
        i = 2
        while len(s) < n:
            cur = 3 - s[-1]
            s += [cur] * s[i]
            i += 1
        return s[:n].count(1)

sol = Solution()
print(sol.magicalString(6), sol.magicalString(1))      # 3 1
assert sol.magicalString(6) == 3 and sol.magicalString(1) == 1

3 1


## 10. Vì sao thuật toán đúng

**Khẳng định 1 (đủ dữ liệu để đọc):** tại mọi thời điểm, `i < len(s)`. Chứng minh quy nạp: khởi đầu `i = 2 < 3 = len(s)`. Mỗi vòng, `i` tăng 1 và `len(s)` tăng `s[i] >= 1`. Vì vậy `len(s) - i` không giảm; nó vẫn dương, nên `s[i]` luôn đã tồn tại khi cần.

**Khẳng định 2 (đúng chuỗi):** mệnh đề "các nhóm của `s` có độ dài `s[0], s[1], s[2], ...` và xen kẽ `1,2,1,2,...`" xác định **duy nhất** từng ký tự theo thứ tự. Mỗi vòng ta ghi đúng nhóm thứ `i` (ký tự xen kẽ, độ dài `s[i]`), tức đúng định nghĩa. Hai nhóm đầu `"1"` (độ dài `s[0] = 1`) và `"22"` (độ dài `s[1] = 2`) được ghi sẵn, và nhóm `2` có độ dài `s[2] = 2` được xử lý trong vòng lặp. Ta đã kiểm chứng tính duy nhất bằng vét cạn (Cách 1 assert đúng một tiền tố hợp lệ).

**Khẳng định 3 (cắt `n`):** tiền tố `n` ký tự của chuỗi dựng được (rộng hơn `n`) chính là `n` ký tự đầu của chuỗi kỳ diệu, vì ta chỉ nối thêm ở **cuối** và không bao giờ sửa phần đã ghi. Cắt `s[:n]` rồi đếm là đúng.

**Cách 3 và 4** chỉ khác cách lưu/đọc cùng một dãy nên đúng theo; ta đối chiếu bằng assert ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `2^n` | `O(n · 2^n)` | `O(n)` |
| 2. Hai con trỏ | `O(n)` | `O(n)` |
| 3. Tiền xử lý + tiền tố | dựng `O(L)`, truy vấn `O(1)` | `O(L)` |
| 4. Bộ sinh lồng nhau | `O(n)` | `O(log n)` |

Tại sao Cách 2 là `O(n)`: mỗi vòng thêm ít nhất 1 phần tử; vòng lặp dừng khi `len(s) >= n`, tức có không quá `n` vòng; mỗi vòng ghi `s[i] <= 2` phần tử nên tổng công việc ghi là `O(n)`. Đếm cuối `O(n)`.

Với `n = 10^5`, Cách 2 chỉ chạy cỡ vài chục mili-giây trong Python.

In [17]:
import time
t0 = time.perf_counter()
kq = dem_hai_con_tro(100000)
print("n = 100000 ->", kq, f"trong {time.perf_counter() - t0:.4f}s")
t0 = time.perf_counter()
kq4 = dem_bo_sinh(100000)
print("Cách 4:", kq4, f"trong {time.perf_counter() - t0:.4f}s")
assert kq == kq4

n = 100000 -> 49972 trong 0.0129s


Cách 4: 49972 trong 0.0244s


## 12. Bộ test

Ta gom các cách giải vào một từ điển, rồi kiểm tra: ví dụ của đề, biên, so với vét cạn, và chuỗi sinh ra có thật sự "tự mô tả" không (hàm kiểm tra tính chất độc lập).

In [18]:
import random

def sinh_chuoi(n):
    """Dựng n ký tự đầu (dùng Cách 2) để kiểm tra tính chất."""
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        s += [3 - s[-1]] * s[i]
        i += 1
    return s[:n]

def tu_mo_ta(s):
    """Hàm kiểm tra độc lập: các độ dài nhóm đầy đủ phải khớp với chính s; nhóm cuối không vượt."""
    return la_tien_to_hop_le(tuple(s))

CACH = {
    "hai_con_tro": dem_hai_con_tro,
    "tien_to": dem_tien_to,
    "bo_sinh": dem_bo_sinh,
}

# 1. Ví dụ đề và edge case nhỏ
for n, kq in [(6, 3), (1, 1), (2, 1), (3, 1), (4, 2), (5, 3)]:
    for ten, f in CACH.items():
        assert f(n) == kq, (ten, n, f(n), kq)
print("ví dụ đề + biên nhỏ: OK")

ví dụ đề + biên nhỏ: OK


In [19]:
# 2. Đối chiếu với oracle vét cạn (n <= 14) và đối chiếu giữa các cách
for n in range(1, 15):
    cho_dung = dem_vet_can(n)[0]
    for ten, f in CACH.items():
        assert f(n) == cho_dung, (ten, n)
print("khớp vét cạn n = 1..14: OK")

for n in range(1, 3001):
    gt = dem_hai_con_tro(n)
    assert dem_tien_to(n) == gt
    if n % 7 == 0:
        assert dem_bo_sinh(n) == gt
print("khớp giữa các cách n = 1..3000: OK")

khớp vét cạn n = 1..14: OK


khớp giữa các cách n = 1..3000: OK


In [20]:
# 3. Tính chất: chuỗi dựng ra thật sự tự mô tả, và đếm 1 bằng hai cách
for n in list(range(1, 300)) + [1000, 5000, 100000]:
    c = sinh_chuoi(n)
    assert tu_mo_ta(c), n
    assert set(c) <= {1, 2}
    assert c[:n].count(1) == dem_hai_con_tro(n)
print("tính chất tự mô tả: OK")

# 4. Ngẫu nhiên có seed, đúng ràng buộc 1 <= n <= 10^5
random.seed(481)
for _ in range(60):
    n = random.randint(1, 100000)
    gt = dem_hai_con_tro(n)
    assert dem_tien_to(n) == gt
    assert Solution().magicalString(n) == gt
print("ngẫu nhiên có seed: OK")

# 5. Tỉ lệ số 1 xấp xỉ 1/2 (tính chất thống kê đã biết của dãy Kolakoski)
print("tỉ lệ số 1 với n = 100000:", dem_hai_con_tro(100000) / 100000)
assert abs(dem_hai_con_tro(100000) / 100000 - 0.5) < 0.01

tính chất tự mô tả: OK


ngẫu nhiên có seed: OK
tỉ lệ số 1 với n = 100000: 0.49972


**Phản ví dụ cho một "mẹo" sai hay gặp:** nhiều bạn đoán "số chữ `1` luôn bằng `ceil(n / 2)`" hoặc `n // 2`. Kiểm tra nhanh:

In [21]:
sai = [n for n in range(1, 40) if dem_hai_con_tro(n) != (n + 1) // 2]
print("các n làm công thức (n+1)//2 sai:", sai[:12], "...")
assert len(sai) > 0
n0 = sai[0]
print(f"ví dụ phản chứng n = {n0}: thật = {dem_hai_con_tro(n0)}, công thức = {(n0 + 1) // 2}")

các n làm công thức (n+1)//2 sai: [3, 9, 11, 12, 13, 15, 19, 21, 27, 32, 34, 39] ...
ví dụ phản chứng n = 3: thật = 1, công thức = 2


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cắt `s[:n]` (nhóm cuối ghi quá `n`)

**Code sai:**
```python
def sai1(n):
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        s += [3 - s[-1]] * s[i]
        i += 1
    return s.count(1)        # SAI: đếm cả phần dư
```

**Vì sao sai:** với `n = 4`, danh sách sau khi dừng là `[1, 2, 2, 1, 1]` (dài 5). Ta chỉ được đếm 4 ký tự đầu.

In [22]:
def sai1(n):
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        s += [3 - s[-1]] * s[i]
        i += 1
    return s.count(1)

print("sai1(4) =", sai1(4), " đúng =", dem_hai_con_tro(4))
assert sai1(4) != dem_hai_con_tro(4)

sai1(4) = 3  đúng = 2


**Code đúng:** thêm `[:n]` như Cách 2.

### Lỗi 2 — Lấy ký tự nhóm theo `s[i]` thay vì xen kẽ với ký tự cuối

**Code sai:**
```python
cur = s[i]                   # SAI: s[i] là ĐỘ DÀI, không phải ký tự của nhóm
```

**Vì sao sai:** `s[i]` cho biết **bao nhiêu** ký tự cần ghi, còn **ký tự nào** phải xen kẽ với nhóm trước ⇒ `3 - s[-1]`. Nhầm hai vai trò này cho kết quả hoàn toàn khác.

In [23]:
def sai2(n):
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        cur = s[i]                       # SAI
        s += [cur] * s[i]
        i += 1
    return s[:n].count(1)

print([sai2(n) for n in range(1, 12)])
print([dem_hai_con_tro(n) for n in range(1, 12)])
assert any(sai2(n) != dem_hai_con_tro(n) for n in range(1, 30))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 2, 3, 3, 4, 4, 4, 5, 5]


### Lỗi 3 — Đặt sai con trỏ đọc ban đầu

**Code sai:** bắt đầu với `s = [1, 2, 2]` nhưng `i = 0` hoặc `i = 1`. Khi đó ta **đọc lại** các giá trị đã dùng để dựng nhóm 0 và nhóm 1 và ghi nhóm thứ 2 với độ dài `s[0] = 1` hoặc `s[1] = 2` thay vì `s[2] = 2` (nhóm thứ 2 phải có độ dài bằng `s[2]`).

In [24]:
def sai3(n):
    s = [1, 2, 2]
    i = 0                                 # SAI: phải là 2
    while len(s) < n:
        s += [3 - s[-1]] * s[i]
        i += 1
    return s[:n].count(1)

print([sai3(n) for n in range(1, 12)])
print([dem_hai_con_tro(n) for n in range(1, 12)])
assert any(sai3(n) != dem_hai_con_tro(n) for n in range(1, 30))

[1, 1, 1, 2, 2, 2, 3, 4, 4, 5, 6]
[1, 1, 1, 2, 3, 3, 4, 4, 4, 5, 5]


### Lỗi 4 — Khởi tạo `s = [1]` rồi thao tác như thường

Với `s = [1]` và `i = 0`: nhóm 0 có độ dài `s[0] = 1` — đã ghi rồi nên ta ghi thêm sẽ **trùng lặp**. Dùng danh sách khởi đầu `[1, 2, 2]` (đã chứa hai nhóm đầu) là cách an toàn; nếu muốn bắt đầu tối giản hơn, phải xử lý riêng `n <= 3`.

**Mẹo kiểm tra nhanh:** luôn thử `n = 1, 2, 3, 4, 5, 6` bằng tay, vì mọi lỗi khởi tạo/cắt `n` lộ ra ở những `n` này.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** đề nói "chuỗi/dãy mô tả chính nó" (self-describing), "độ dài nhóm tạo ra chuỗi". Gợi ý ngay: **hai con trỏ trên cùng một mảng**, một ghi, một đọc.
- **Điều kiện an toàn:** luôn kiểm tra con trỏ đọc **không vượt** con trỏ ghi (`i < len(s)`).
- **Luôn cắt `[:n]`** khi bước nhảy có thể vượt quá giới hạn.
- **Nhiều truy vấn trên dãy cố định** ⇒ tiền xử lý một lần + mảng tiền tố (Cách 3).
- **Bài toán dựng chuỗi theo quy tắc tự tham chiếu** thường là mô phỏng thuần túy; đừng thử vét cạn.
- Dãy này có tên **dãy Kolakoski** (OEIS A000002 với bảng chữ `{1,2}`).

## 15. Biến thể

1. **Đếm số `2`** trong `n` ký tự đầu: `n - dem(1)`.
2. **Bảng chữ khác** `{a, b}`, ví dụ `{1, 3}`: dựng với `s = [1]` và quy tắc `cur = a + b - cur` (xen kẽ), cần xử lý khởi tạo riêng. Dãy Kolakoski tổng quát.
3. **Trả về ký tự thứ `k`** của chuỗi kỳ diệu: dùng Cách 2 hoặc Cách 4.
4. **Kiểm tra** một chuỗi 1/2 cho trước có phải **tiền tố** của chuỗi kỳ diệu không (so sánh với `sinh_chuoi`).
5. **Nếu `n` lên tới `10^9`**: Cách 4 cho `O(n)` thời gian nhưng chỉ `O(log n)` bộ nhớ, còn tiền tố lưu `O(n)` sẽ không nổi. (Dãy Kolakoski chưa biết công thức đóng, nên vẫn phải mô phỏng.)

In [25]:
# Biến thể: đếm số 2, và ký tự thứ k
def dem_so_2(n):
    return n - dem_hai_con_tro(n)

def ky_tu_thu_k(k):             # k tính từ 1
    return sinh_chuoi(k)[k - 1]

print(dem_so_2(6), ky_tu_thu_k(6), [ky_tu_thu_k(k) for k in range(1, 13)])
assert dem_so_2(6) == 3 and ky_tu_thu_k(6) == 2

3 2 [1, 2, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi chuỗi 1/2, lọc theo định nghĩa | `O(n·2^n)` | `O(n)` | chỉ làm oracle kiểm thử, `n ≲ 14` |
| 2. Hai con trỏ | ghi/đọc trên cùng một danh sách | `O(n)` | `O(n)` | cách nộp chuẩn, dễ viết nhất |
| 3. Tiền xử lý + prefix | dựng 1 lần, truy vấn `O(1)` | `O(L)` + `O(1)`/truy vấn | `O(L)` | nhiều truy vấn cùng dãy |
| 4. Bộ sinh lồng nhau | đọc từ bản sao lùi sau của chính nó | `O(n)` | `O(log n)` | khi bộ nhớ quan trọng, hoặc `n` rất lớn |

**Khuyên dùng khi thi:** Cách 2.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `dem_hai_con_tro(8)`, `dem_hai_con_tro(10)` và `sinh_chuoi(8)`; sau đó chạy để kiểm tra.

In [26]:
# Bài 1: ghi dự đoán của bạn vào đây rồi chạy để so sánh
du_doan_8 = None
du_doan_10 = None
du_doan_chuoi = None
print("Thực tế:", dem_hai_con_tro(8), dem_hai_con_tro(10), sinh_chuoi(8))

Thực tế: 4 5 [1, 2, 2, 1, 1, 2, 1, 2]


**Bài 2 (tính tay).** Viết tay 15 ký tự đầu của chuỗi kỳ diệu theo quy tắc của mục 4, rồi đếm số `1` trong 15 ký tự đó. Không chạy code trước khi có đáp án.

**Bài 3 (điền TODO).** Hoàn thiện hàm `do_dai_nhom_todo(s)` trả về danh sách độ dài các nhóm (không dùng `groupby`).

In [27]:
def do_dai_nhom_todo(s):
    # TODO: duyệt s, giữ biến dem; khi ký tự đổi thì thêm dem vào kết quả và đặt lại dem = 1
    #       nhớ thêm nhóm cuối cùng sau vòng lặp
    pass

kq_bai3 = None       # sau khi viết xong, gán kq_bai3 = do_dai_nhom_todo("1221121")

**Bài 4 (tự viết lại).** Không nhìn Cách 2, hãy tự viết hàm `dem_tu_viet(n)` trả về số chữ `1` trong `n` ký tự đầu. Bạn phải xử lý: khởi tạo, con trỏ đọc, xen kẽ ký tự, cắt `[:n]`.

In [28]:
def dem_tu_viet(n):
    # TODO: tự viết
    pass

**Bài 5 (biến thể / mini challenge).** Dãy Kolakoski với bảng chữ `{1, 3}`: gom nhóm, độ dài các nhóm nối lại đúng chuỗi, bắt đầu bằng `1`. Viết `dem_1_va_3(n)` trả về số chữ `1` trong `n` ký tự đầu. Gợi ý ở mục 18.

In [29]:
def dem_1_va_3(n):
    # TODO: khởi tạo, rồi mô phỏng hai con trỏ như Cách 2 nhưng với bảng chữ {1, 3}
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):**
  - Bài 1: dựng chuỗi theo mục 4: `1`, `22`, `11`, `2`, `1`, `22`, `1`, `22`.
  - Bài 3: dùng một biến `dem` và một vòng `for` so sánh `s[k]` với `s[k-1]`.
  - Bài 5: bảng chữ `{1,3}` thì ký tự xen kẽ là `4 - cur`.
- **Mức 2 (vừa):**
  - Bài 2: sau 12 ký tự `122112122122`, ghi tiếp nhóm `11` (độ dài `s[6] = 1`? hãy đọc đúng `s[i]` ở con trỏ), rồi `2`; tự đối chiếu bằng cách gom nhóm lại chuỗi vừa viết.
  - Bài 4: bốn thành phần: danh sách khởi đầu `[1,2,2]`, `i = 2`, `cur = 3 - s[-1]`, `s[:n]`.
  - Bài 5: nhóm 0 là `1` (độ dài `s[0] = 1`); nhóm 1 gồm ký tự `3`, độ dài `s[1]` mà `s[1]` chính là ký tự đầu của nhóm 1, tức `3`; nên `s = [1, 3, 3, 3]`.
- **Mức 3 (gần đáp án):**
  - Bài 4: copy Cách 2, đổi tên hàm.
  - Bài 5: khởi tạo `s = [1, 3, 3, 3]`, `i = 2`, vòng `while len(s) < n: s += [4 - s[-1]] * s[i]; i += 1`; trả `s[:n].count(1)`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert sinh_chuoi(8) == [1, 2, 2, 1, 1, 2, 1, 2]
assert dem_hai_con_tro(8) == 4          # 1 2 2 1 1 2 1 2 -> bốn chữ 1? đếm: vị trí 0,3,4,6 => 4
assert dem_hai_con_tro(10) == 5         # 1 2 2 1 1 2 1 2 2 1 -> vị trí 0,3,4,6,9 => 5

# Bài 2: 15 ký tự đầu
c15 = sinh_chuoi(15)
print("".join(map(str, c15)), "->", c15.count(1))
assert "".join(map(str, c15)) == "122112122122112"
assert c15.count(1) == 7

122112122122112 -> 7


In [31]:
# Bài 3
def do_dai_nhom_dap_an(s):
    kq = []
    dem = 1
    for k in range(1, len(s)):
        if s[k] == s[k - 1]:
            dem += 1
        else:
            kq.append(dem)
            dem = 1
    kq.append(dem)
    return kq

assert do_dai_nhom_dap_an("1221121") == [1, 2, 2, 1, 1]
assert do_dai_nhom_dap_an("1") == [1]
assert do_dai_nhom_dap_an("1221121") == do_dai_cac_nhom("1221121")

# Bài 4
def dem_tu_viet_dap_an(n):
    s = [1, 2, 2]
    i = 2
    while len(s) < n:
        s += [3 - s[-1]] * s[i]
        i += 1
    return s[:n].count(1)

for n in range(1, 400):
    assert dem_tu_viet_dap_an(n) == dem_hai_con_tro(n)

In [32]:
# Bài 5: Kolakoski {1, 3}
def dem_1_va_3_dap_an(n):
    s = [1, 3, 3, 3]
    i = 2
    while len(s) < n:
        s += [4 - s[-1]] * s[i]
        i += 1
    return s[:n].count(1)

c = [1, 3, 3, 3]
i = 2
while len(c) < 30:
    c += [4 - c[-1]] * c[i]
    i += 1
print(c[:30])
# kiểm tra tính chất tự mô tả: các nhóm đầy đủ phải có độ dài bằng đúng các phần tử đầu
nhom = [len(list(g)) for _, g in groupby(c[:30])]
assert nhom[:-1] == c[:len(nhom) - 1]
assert dem_1_va_3_dap_an(4) == 1
assert dem_1_va_3_dap_an(10) == c[:10].count(1)
print("Bài 5 OK")

[1, 3, 3, 3, 1, 1, 1, 3, 3, 3, 1, 3, 1, 3, 3, 3, 1, 1, 1, 3, 3, 3, 1, 3, 3, 3, 1, 3, 3, 3]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hai con trỏ ghi/đọc trên cùng mảng):**

```python
class Solution:
    def magicalString(self, n: int) -> int:
        s = [1, 2, 2]          # hai nhóm đầu đã biết
        i = 2                  # con trỏ đọc: độ dài của nhóm kế tiếp
        while len(s) < n:
            cur = 3 - s[-1]    # ký tự xen kẽ
            s += [cur] * s[i]  # ghi s[i] ký tự
            i += 1
        return s[:n].count(1)  # nhớ cắt n
```

**Checklist trước khi submit:**

- [ ] `s = [1, 2, 2]` và `i = 2` (không phải `i = 0`).
- [ ] Ký tự nhóm mới là `3 - s[-1]`, **không** phải `s[i]`.
- [ ] Độ dài nhóm là `s[i]`.
- [ ] Đếm trên `s[:n]`, không đếm cả danh sách.
- [ ] Đã thử `n = 1, 2, 3, 4, 6`.
- [ ] Độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ.